# Greath North Run data processing

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
raw_df = pd.read_excel("2024 Data Challenge Raw Data.xlsx")
raw_df.shape

## Row descriptions
|Variable |	Definition |
| :-: | :- |
| club |	The club the runner is a part of 
|position |	The position the runner finished in 
|time |	The time the runner took to finish the race
|id |	The runners id
|gender	| The runners gender
|trained_10_week |	The average number of times per week the runner trained 10 or more weeks ago
|trained_im	| The average number of times per week the runner trained less than e weeks ago
|has_trainer |	Whether the runner has a trainer or not
|cadence |	The average number of steps the runner took per minute
|age |	The age of the runner
|bmi |	The bmi of the runner
|n_marathons_run |	The number of marathons the runner has run before the race
|bib_colour |	The runners bib colour
|VO2_max |	The VO2 max of the runner
|heart_rate	| The resting heart rate of the runner
|shoe_size	| The runner's shoe size


NOTE:  
*Bib colours have meaning:  
Orange: 

In [ ]:
raw_df.head()

#figure out unnamed column name- external sources?
#fix club names for all the records, fill up until you find text

In [ ]:
#no of missing values in each column 
raw_df.isnull().sum()

#autofill club information for all the other missing rows, boundaries to be defined from the excel spreadsheet 
#strategy for rows with missing time, impute/discard?
#

In [ ]:
raw_df.describe()

#trained 10 weeks, verify -999
#max BMI 999, fix that
#fix -9 in min for n_marathons_run
#fix max for n_marathons_run. is it normal for someone to run 1mil marathons? anomaly- discard

In [ ]:

print(raw_df.bib_colour.value_counts())

In [ ]:
# Cleaning steps
# [x] remove non existant: position, id, final time
# [x] trained_10_week & trained_im: replace negative with zero
# [x] has_trainer: one hot (already one hot)
# [x] bib colour: one hot
# [x] bmi: remove outliers
# [x] n_runs: remove negative and outliers
# [x] sex: one hot

df = raw_df
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)
    # Source - https://stackoverflow.com/a/49554761
    # Posted by Adil Warsi, modified by community. See post 'Timeline' for change history
    # Retrieved 2026-09-22, License - CC BY-SA 4.0
    df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)
df.dropna(subset=['time'], inplace=True)
df['time'] = pd.to_timedelta(df['time']).dt.total_seconds().astype(int)

# clean missing and negative values
df['trained_10_week'] = df['trained_10_week'].mask(df['trained_10_week'] < 0, 0)
df['trained_10_week'] = df['trained_10_week'].mask(df['trained_10_week'].isna(), 0)
df['trained_im'] = df['trained_im'].mask(df['trained_im'] < 0, 0)
df['trained_im'] = df['trained_im'].mask(df['trained_im'].isna(), 0)

# missing, negative and outliers
df['n_marathons_run'] = df['n_marathons_run'].mask(df['n_marathons_run'] < 0, 0)
df['n_marathons_run'] = df['n_marathons_run'].mask(df['n_marathons_run'].isna(), 0)
# todo: find and remove outliers

# multi hot encoding
df['bib_colour_multi'], _ = pd.factorize(df['bib_colour'])
df['gender_multi'], _ = pd.factorize(df['gender'])

bmi_outliers = df['bmi'] == 999.0
print(bmi_outliers.value_counts())  # about 568 such cases so there is data error
df['bmi'] = df['bmi'].mask(df['bmi'] == 999.0, df['bmi'].mean())

# drop missing values cause they have strong correlation
df.dropna(subset=['VO2_max', 'heart_rate'])

# drop rows for people who have run more than =1000 marathons 
df = df.drop(df[df["n_marathons_run"] >= 700].index)
df.describe()

In [ ]:
df.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 1. Define your exact list of columns
columns = [
    "position",
    "time",
    "trained_10_week",
    "trained_im",
    "has_trainer",
    "cadence",
    "age",
    "bmi",
    "n_marathons_run",
    "VO2_max",
    "heart_rate",
    "shoe_size",
    "bib_colour_multi",
    "gender_multi",
]

# 2. Compute the Pearson correlation matrix on your DataFrame
corr = df[columns].corr(method="pearson")

# 3. Generate a boolean mask for the upper triangle
# This hides the redundant mirror reflection and keeps the plot clean
mask = np.triu(np.ones_like(corr, dtype=bool))

# 4. Set up the matplotlib figure size (larger to fit all 14 variables)
plt.figure(figsize=(14, 12))

# 5. Plot the triangular heatmap
sns.heatmap(
    corr,
    mask=mask,
    annot=True,  # Displays numbers inside the squares
    fmt=".2f",  # Rounds visual annotations to 2 decimal places
    cmap="coolwarm",  # Red for positive correlation, blue for negative
    vmin=-1,
    vmax=1,  # Standardizes the color scale limits
    square=True,  # Forces cells to be perfect squares
    linewidths=0.5,  # Adds clean white borders between cells
    cbar_kws={"shrink": 0.8},  # Shrinks colorbar slightly for visual balance
)

# 6. Add a title and tweak layout so labels don't get cut off
plt.title("Running Dataset: Pearson Correlation Matrix", fontsize=16, pad=20)
plt.tight_layout()
plt.show()


#Since there is a very strong correlation between the VO2_max and heart_rate, populating it with an average will affect the reliabilioty of the model to accurately predict the performance of a runner, and since the unique number of rows with the intersection of both these values is less than 10% of the data, we have chosen to eliminate it


In [ ]:
(df["bib_colour"][df["cadence"] == 1.00000]).value_counts()

In [ ]:

average_timings = df.groupby(['bib_colour', 'gender'])['time'].mean().reset_index()
average_timings.columns = ['Bib Color', 'Gender', 'Average Seconds']

# 2. Convert the average seconds into HH:MM:SS format
average_timings['Average HH:MM:SS'] = pd.to_timedelta(average_timings['Average Seconds'].round(), unit='s')
average_timings['Average HH:MM:SS'] = average_timings['Average HH:MM:SS'].apply(lambda x: str(x).split()[-1])

print(average_timings)

In [ ]:
# a.
# - clean and why you clean what you clean
# - mean, std, min, max, basically .describe()
# - yap about distribution (maybe pair plot)

# b.
# - pearson correlation matrix

# c.
# get rid of position
# inc VO2_max, heart rate, times trained both variables, bib colour, n_marathons_run (mandatory)
# gender